<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 4: Pandas, transformaciones</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

<h2 style='font-weight:bold; color:#003366;'>Pandas: clase taller</h2>

La clase funciona como un mini proyecto de análisis con dos archivos distintos.

**Cómo se organiza.** La clase se divide en dos grupos. Cada grupo recibe uno de los dos casos que están
más abajo. El trabajo es individual: cada persona escribe su propio código y saca sus propias
conclusiones. Al final hay una puesta en común donde cada grupo le cuenta al otro qué encontró, porque
la otra mesa no vio esos datos y tiene que poder entender el hallazgo sin haber tocado el archivo.

<h2 style='font-weight:bold; color:#003366;'>1. Introducción: Demo</h2>

Se usa un dataset distinto al de los casos, `ventas_ecommerce.csv`, para ver el recorrido
completo una vez antes de arrancar: cargar, mirar, construir una columna y sacar una conclusión.

Son cuatro pasos y son siempre los mismos, sin importar el dataset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


def ruta_datos(nombre_archivo, subcarpeta=""):
    """Busca el archivo en las ubicaciones habituales del curso, incluida Colab."""
    candidatas = [
        Path("../Datos") / subcarpeta / nombre_archivo,
        Path("Datos") / subcarpeta / nombre_archivo,
        Path("/content/Datos") / subcarpeta / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")

Primer paso: **cargar el archivo**, si se sabe algo se utiliza, por ejemplo existe una columna fecha. 

Las fechas se declaran en la carga
con `parse_dates`, porque una columna de fechas leída como texto no permite restar días ni pedir el mes.
El resto de los tipos se revisa después.

In [2]:
ventas = pd.read_csv(ruta_datos("ventas_ecommerce.csv"), parse_dates=["fecha"])

print(f"Filas {ventas.shape[0]} y columnas {ventas.shape[1]}:")
ventas.head(3)

Filas 8637 y columnas 11:


,id_orden,fecha,categoria,producto,region,canal,unidades,precio_unitario,descuento,costo_envio,cliente_nuevo
0,ORD-0000-00000,2024-01-01,Hogar,Silla ergonomica,Montevideo,Marketplace,1,"5,634.83",0.10,224.99,False
1,ORD-0000-00001,2024-01-01,Indumentaria,Zapatillas,Colonia,Web,3,"2,315.23",0.00,173.71,True
2,ORD-0000-00002,2024-01-01,Indumentaria,Pantalon,Montevideo,Marketplace,2,"2,495.15",0.10,233.84,False


Segundo paso: **exploración muy básica de datos**. 

1. Cuántas filas hay
2. Qué tipo tiene cada columna
3. Cuántos faltantes hay
4. En qué rango se mueven los números. 
5. Distribución de variables categóricas.

La mayoría de los errores de análisis se evitan acá.

In [10]:
print("1. Tipos de dato")
print(ventas.dtypes)

print("\n2. Faltantes por columna")
print(ventas.isna().sum().loc[lambda serie: serie > 0])

print("\n3. Rango de fechas:", ventas["fecha"].min().date(), "a", ventas["fecha"].max().date())

print("\n4. Escala de las columnas numéricas")
display(ventas[["unidades", "precio_unitario", "descuento", "costo_envio"]].describe().round(2))

print("\n4. Distribución de las columnas categóricas")
display(ventas.describe(include="object", exclude=["number"]).round(2))

1. Tipos de dato
id_orden                     object
fecha                datetime64[ns]
categoria                    object
producto                     object
region                       object
canal                        object
unidades                      int64
precio_unitario             float64
descuento                   float64
costo_envio                 float64
cliente_nuevo                  bool
facturacion                 float64
margen_neto                 float64
envio_no_rentable            object
dtype: object

2. Faltantes por columna
Series([], dtype: int64)

3. Rango de fechas: 2024-01-01 a 2026-06-30

4. Escala de las columnas numéricas


,unidades,precio_unitario,descuento,costo_envio
count,"8,637.00","8,637.00","8,637.00","8,637.00"
mean,2.48,"6,688.14",0.08,303.04
std,1.12,"6,703.11",0.09,106.32
min,1.00,392.44,0.00,-36.29
25%,1.00,"2,269.85",0.00,221.91
50%,2.00,"4,074.07",0.10,294.37
75%,3.00,"6,303.40",0.15,378.34
max,4.00,"28,847.46",0.25,688.44



4. Distribución de las columnas categóricas


,id_orden,categoria,producto,region,canal,envio_no_rentable
count,8637,8637,8637,8637,8637,8637
unique,8637,5,19,6,4,2
top,ORD-0000-00000,Indumentaria,Zapatillas,Montevideo,Web,No
freq,1,2326,598,3267,3554,8484


Tercer paso: **construir variables de negocio**.

En el archivo hay unidades, precio y descuento.
No hay facturación. La facturación es lo que le interesa a cualquiera que mire este dataset y hay que
calcularla.

Las operaciones entre columnas se escriben una sola vez y pandas las aplica a todas las filas.

In [ ]:
# El archivo no trae el costo de la mercadería, así que el margen se estima con un supuesto.
MARGEN_BRUTO_SUPUESTO = 0.35 # Supuesto de margen bruto sobre facturación, antes de costos de envío y descuentos.

ventas["facturacion"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento"])
ventas["margen_neto"] = ventas["facturacion"] * MARGEN_BRUTO_SUPUESTO - ventas["costo_envio"]

print(f"Facturación total: $ {ventas['facturacion'].sum():,.0f}")
print(f"Compra promedio: $ {ventas['facturacion'].mean():,.0f}")
print(f"Compra mediana: $ {ventas['facturacion'].median():,.0f}")

Facturación total: $ 130,245,617
Compra promedio: $ 15,080
Compra mediana: $ 8,300


El promedio de facturación por orden es casi cuatro veces la mediana.

Antes de correr la celda siguiente responda: si se le pide a este dataset
la compra típica de una orden, **cuál de los dos números corresponde usar en el informe, y por qué**
la diferencia es tan grande.

In [5]:
# El 10% de las órdenes más grandes, y qué porcentaje de la facturación total explica
corte = ventas["facturacion"].quantile(0.90)
grandes = ventas[ventas["facturacion"] > corte]

print(f"Órdenes por encima del percentil 90: {len(grandes):,} de {len(ventas):,}")
print(f"Explican el {grandes['facturacion'].sum() / ventas['facturacion'].sum():.1%} "
      f"de la facturación total")

Órdenes por encima del percentil 90: 864 de 8,637
Explican el 40.7% de la facturación total


Cuarto paso: **Clasificar filas según una regla de negocio**. `np.where` toma una condición y
devuelve un valor cuando se cumple y otro cuando no, para toda la columna a la vez.

Y una vez que la clasificación existe, `value_counts()` sobre las filas que cumplen la condición dice
de dónde viene el problema. Esa combinación de filtro más recuento es la herramienta de investigación
más barata que tiene pandas.

In [6]:
ventas["envio_no_rentable"] = np.where(ventas["margen_neto"] < 0, "Si", "No")

print("Órdenes con margen neto negativo:",
      f"{(ventas['envio_no_rentable'] == 'Si').sum():,}")

print("\nDe qué región vienen esas órdenes:")
print(ventas.loc[ventas["envio_no_rentable"] == "Si", "region"].value_counts().head(5))

print("\nY qué peso tiene cada región en el total de órdenes:")
print(ventas["region"].value_counts(normalize=True).head(5).round(1)) # con Normalize=True se obtiene la proporción de cada categoría

Órdenes con margen neto negativo: 153

De qué región vienen esas órdenes:
region
Canelones         39
Interior resto    34
Maldonado         31
Salto             20
Montevideo        15
Name: count, dtype: int64

Y qué peso tiene cada región en el total de órdenes:
region
Montevideo       0.40
Canelones        0.20
Interior resto   0.10
Maldonado        0.10
Colonia          0.10
Name: proportion, dtype: float64


<h2 style='font-weight:bold; color:#003366;'>2. Los dos casos</h2>

Los dos casos comparten la estructura: hay un archivo principal con una fila por unidad de análisis,
hay una pregunta de negocio que no se contesta con las columnas que vinieron, y hay algo raro adentro.

<h3 style='font-weight:bold; color:#0033BB;'>Caso A: Banco Litoral: rentabilidad de la banca minorista</h3>

**Contexto.** Banco Litoral es un banco mediano con catorce sucursales. El comité de dirección quiere
subir la rentabilidad de la banca minorista y puso sobre la mesa una propuesta concreta: cerrar las
sucursales del interior, porque atienden pocos clientes y cuestan lo mismo que las de Montevideo.

**El pedido.** El área de riesgo pide una lectura de la cartera de clientes antes de decidir. La pregunta
que hay que contestar hoy es: *cuánto deja cada cliente por año, y hay clientes que le cuestan plata al
banco.*

**El archivo.** `Datos/casos/banca_clientes.csv`, una fila por cliente, con el saldo promedio de
depósitos, el saldo promedio de préstamos, las tasas pactadas y las comisiones mensuales.

**Glosario:** 

- La *tasa activa* es la que el banco cobra por un préstamo. 
- La *tasa pasiva* es la que el banco paga por un depósito. 
- El *spread* es la diferencia entre las dos y es de donde sale el margen financiero. 
- La *tasa de transferencia interna* es el precio al que la tesorería del banco le
compra los depósitos y le vende los fondos a cada sucursal: sirve para saber cuánto aporta cada producto
por separado, y en este banco está fijada en 9,5% anual.

<h3 style='font-weight:bold; color:#BB3300;'>Caso B: Sierra Norte S.A.: dónde se pierde el margen</h3>

**Contexto.** Sierra Norte S.A. es una distribuidora de equipamiento agroindustrial con 112 clientes
empresa. Los últimos dos años y medio facturó siempre parecido, pero el dueño observa dos cosas que no
cierran: el resultado del ejercicio bajó, y la plata en la cuenta bajó más todavía.

**El pedido.** Contrató una asesoría financiera. La pregunta que hay que contestar hoy es: *qué parte de
lo que se factura llega efectivamente al resultado, y dónde se está perdiendo.*

**El archivo.** `Datos/casos/empresa_facturas.csv`, una fila por factura emitida entre enero de 2024 y
junio de 2026, con la línea de producto, el precio, el descuento otorgado, el costo de la mercadería, el
flete, el vendedor y la fecha de cobro cuando la factura fue cobrada.

**Vocabulario mínimo.** El *margen bruto* es la facturación menos el costo de la mercadería vendida. El
*margen neto* le resta además los costos directos de la venta, acá el flete. Los *días de cobro* son los
que pasan entre la emisión de la factura y el cobro: mientras la factura no se cobra, la empresa le está
prestando plata al cliente sin interés.

Cada grupo corre solamente la celda de su caso. Correr las dos no molesta, pero duplica el trabajo de leer salidas que no se van a usar.

In [7]:
# Caso A: Banco Litoral
clientes = pd.read_csv(ruta_datos("banca_clientes.csv", "casos"), parse_dates=["fecha_alta"])

TASA_TRANSFERENCIA = 9.5  # tasa de transferencia interna anual, en porcentaje

print("Filas y columnas:", clientes.shape)
clientes.head(3)

Filas y columnas: (855, 11)


,id_cliente,fecha_alta,segmento,sucursal,saldo_deposito_prom,saldo_prestamo_prom,tasa_activa_anual,tasa_pasiva_anual,comisiones_mes,productos_contratados,estado
0,CL0659,2015-08-30,Empresa,SUC03,"1,102,531.13","997,791.12",19.61,7.99,"7,495.94",4,Activo
1,CL0169,2021-02-28,Personal,SUC11,"26,472.83",0.00,NaN,4.18,470.01,2,Inactivo
2,CL0165,2022-09-30,Personal,SUC08,"99,076.84","100,726.80",42.39,2.97,548.78,2,Activo


In [8]:
# Caso B: Sierra Norte S.A.
facturas = pd.read_csv(ruta_datos("empresa_facturas.csv", "casos"),
                       parse_dates=["fecha_emision", "fecha_cobro"])

TIPO_CAMBIO = 41.0  # provisorio: en la clase 5 se reemplaza por la cotización de cada mes

print("Filas y columnas:", facturas.shape)
facturas.head(3)

FileNotFoundError: No se encontró empresa_facturas.csv.

<h2 style='font-weight:bold; color:#003366;'>Etapa 1: análisis básico de archivo <span style='font-weight:normal; font-size:70%;'>(10 minutos)</span></h2>

Antes de calcular nada hay que poder describir el archivo en voz alta. La etapa termina cuando se puede
contestar sin volver a mirar: cuántas filas hay, cuál es la unidad de análisis, qué columnas tienen
faltantes y qué rango cubren las fechas y los montos.

Dos preguntas para anotar la respuesta en un comentario para la puesta en común:

1. Cuál es la unidad de análisis del archivo, o sea qué representa exactamente una fila.
2. La columna con más faltantes, ese faltante es un error de carga o significa algo.

In [ ]:
tabla = clientes        # el DataFrame del caso que le toca a la mesa
columnas_clave = ["saldo_deposito_prom", "saldo_prestamo_prom", "tasa_activa_anual", 
                  "tasa_pasiva_anual", "comisiones_mes", "productos_contratados"]   # las columnas numéricas que importan para la pregunta de negocio

print("1. Filas y columnas:", tabla.shape)

print("\n2. Tipos de dato")
print(tabla.dtypes)

print("\n3. Faltantes por columna")
print(tabla.isna().sum().loc[lambda serie: serie > 0])

print("\n4. Escala de lo que importa")
display(tabla[columnas_clave].describe().round(2))

print("\n5. Distribución de las columnas categóricas")
display(tabla.describe(include="object", exclude=["number"]).round(2))

1. Filas y columnas: (855, 11)

2. Tipos de dato
id_cliente                       object
fecha_alta               datetime64[ns]
segmento                         object
sucursal                         object
saldo_deposito_prom             float64
saldo_prestamo_prom             float64
tasa_activa_anual               float64
tasa_pasiva_anual               float64
comisiones_mes                  float64
productos_contratados             int64
estado                           object
dtype: object

3. Faltantes por columna
tasa_activa_anual    427
dtype: int64

4. Escala de lo que importa


,saldo_deposito_prom,saldo_prestamo_prom,tasa_activa_anual,tasa_pasiva_anual,comisiones_mes,productos_contratados
count,855.00,855.00,428.00,855.00,855.00,855.00
mean,"762,649.46","1,289,263.08",33.56,4.69,"2,447.77",3.93
std,"2,338,036.25","13,580,979.39",11.89,1.84,"5,410.30",1.95
min,"-175,986.47",0.00,0.80,1.39,49.11,1.00
25%,"44,928.34",0.00,24.34,3.37,260.99,2.00
50%,"121,857.29","13,496.17",33.25,4.20,568.84,4.00
75%,"539,549.26","416,146.94",45.12,5.64,"2,202.74",6.00
max,"41,250,000.00","388,400,000.00",55.18,10.89,"59,991.41",7.00



5. Distribución de las columnas categóricas


,id_cliente,segmento,sucursal,estado
count,855,855,855,855
unique,841,5,14,2
top,CL0573,Personal,SUC01,Activo
freq,2,358,143,782


<h2 style='font-weight:bold; color:#003366;'>Etapa 2: limpiar lo que está mal <span style='font-weight:normal; font-size:70%;'>(10 minutos)</span></h2>

Los dos archivos vienen de un sistema de gestión que en algún momento importó datos dos veces. Además hay
valores que son imposibles: no ilegales ni raros, imposibles.

Tres cosas para revisar en esta etapa:

1. **Duplicados.** Cuántas filas están repetidas de forma exacta, y cuántos identificadores aparecen más
   de una vez. No es la misma pregunta y la diferencia importa: dos filas con el mismo identificador y
   datos distintos son un problema mucho peor que dos filas idénticas.
2. **Valores imposibles.** En el caso A, un saldo promedio de depósitos no puede ser negativo, porque un
   giro en descubierto se registra como préstamo y no como depósito. En el caso B, una factura no puede
   cobrarse antes de emitirse.
3. **El efecto sobre el número principal.** Antes de borrar, medir cuánto cambia el total. Un duplicado
   que no mueve el resultado es una nota al pie. Uno que lo mueve tres puntos es un hallazgo.

**Importante**: nunca borrar sin contar cuantas filas implica ese cambio y dejar escrito cuántas filas se fueron y con qué
criterio.

In [ ]:
# Caso A
print("Filas totales:", len(clientes))
print("Filas duplicadas exactas:", clientes.duplicated().sum())
print("Identificadores repetidos:", clientes.duplicated(subset="id_cliente").sum())

# Cuánto se infla la cartera de préstamos por las filas repetidas
cartera_con_duplicados = clientes["saldo_prestamo_prom"].sum()
clientes_limpios = clientes.drop_duplicates().copy()
cartera_limpia = clientes_limpios["saldo_prestamo_prom"].sum()

print(f"\nCartera de préstamos informada: $ {cartera_con_duplicados:,.0f}")
print(f"Cartera de préstamos real: $ {cartera_limpia:,.0f}")
print(f"Diferencia: $ {cartera_con_duplicados - cartera_limpia:,.0f}")

# Valores imposibles: completar la condición
descubiertos = clientes_limpios[
    clientes_limpios["saldo_deposito_prom"] < 0
]
print("\nCuentas con saldo de depósito imposible:", len(descubiertos))
display(descubiertos[["id_cliente", "segmento", "sucursal", "saldo_deposito_prom"]])

Filas totales: 855
Filas duplicadas exactas: 14
Identificadores repetidos: 14

Cartera de préstamos informada: $ 1,102,319,937
Cartera de préstamos real: $ 1,095,996,270
Diferencia: $ 6,323,666

Cuentas con saldo de depósito imposible: 2


,id_cliente,segmento,sucursal,saldo_deposito_prom
500,CL0628,Pyme,SUC01,"-116,917.79"
713,CL0226,Personal,SUC11,"-175,986.47"


In [ ]:
# Caso B
print("Filas totales:", len(facturas))
print("Filas duplicadas exactas:", facturas.duplicated().sum())
print("Números de factura repetidos:", facturas.duplicated(subset="id_factura").sum())

facturas_limpias = facturas.drop_duplicates().copy()

# Valores imposibles: facturas con fecha de cobro anterior a la emisión.
# La resta de dos columnas de fecha da un objeto de tipo timedelta.
# Con el accesor .dt se le pide la cantidad de días.
facturas_limpias["dias_cobro"] = (facturas_limpias["fecha_cobro"]
                                  - facturas_limpias["fecha_emision"]).dt.days

imposibles = facturas_limpias[____]
print("\nFacturas cobradas antes de emitirse:", len(imposibles))
print(imposibles[["id_factura", "fecha_emision", "fecha_cobro", "dias_cobro"]])

# Y la decisión sobre las unidades faltantes, escrita en un comentario:
# ¿se descartan esas facturas, se imputan con algún valor, o se dejan como están?

<h2 style='font-weight:bold; color:#003366;'>Etapa 3: construir la variable que falta <span style='font-weight:normal; font-size:70%;'>(15 minutos)</span></h2>

La pregunta de negocio de cada caso no se puede contestar con
ninguna columna del archivo: hay que construirla.

**Caso A: cuánto deja cada cliente por año.** Se arma en tres pedazos que después se suman:

- lo que aporta el préstamo: saldo por la diferencia entre la tasa activa y la tasa de
  transferencia interna.
- lo que aporta el depósito: saldo por la diferencia entre la tasa de transferencia interna y
  la tasa pasiva.
- lo que aportan las comisiones: que están mensuales y el resultado se pide anual.

El pedazo del depósito es el que suele sorprender: un cliente que solo tiene plata depositada igual deja
margen, porque el banco toma esos fondos a una tasa baja y los usa para prestar. Ese es exactamente el
sentido de la tasa de transferencia interna.

**Caso B: cuánto queda de lo que se factura.** También en pedazos:

- La facturación neta de descuento: las unidades por el precio menos el descuento.
- el costo de la mercadería vendida: las unidades por el costo unitario.
- el margen bruto: la resta de los dos anteriores.
- el margen neto: le saca el flete.

Y una complicación real: el 30% de las facturas está en dólares. Sumar pesos con dólares en la misma
columna da un número sin significado. Hoy se resuelve con la constante `TIPO_CAMBIO` y en la clase que
viene con la cotización de cada mes.

Las dos celdas de abajo están vacías a propósito. Los nombres de las columnas están en la celda de carga
y las fórmulas están escritas arriba en palabras.

> **LA SINTAXIS QUE HACE FALTA**
>
> Las operaciones se escriben entre columnas enteras, no fila por fila. `tabla["a"] * tabla["b"]` devuelve
> una columna nueva con el producto de cada par, y se puede asignar directo con `tabla["c"] = ...`.
>
> Para elegir un valor según una condición existe `np.where(condicion, valor_si, valor_no)`, que también
> opera sobre la columna completa. Sirve, por ejemplo, para convertir la moneda: si es dólares se
> multiplica por el tipo de cambio, si no se deja como está.

In [ ]:
# Caso A: construir el ingreso anual por cliente.
# Ojo con dos cosas: las tasas están en porcentaje y hay clientes sin préstamo, con tasa activa faltante.
# Multiplicar por un valor faltante propaga el faltante a todo el resultado (averiguar en pandas cómo se rellenan los nulos)

# 1. Relleno de nulos en la tasa activa:
# Los clientes sin préstamos tienen tasa_activa_anual en NaN. Rellenamos con 0 
# para evitar que la multiplicación devuelva NaN (propagación de nulos).
clientes_limpios["tasa_activa_anual"] = clientes_limpios["tasa_activa_anual"].fillna(0) # fillna() rellena los nulos

# 2. Cálculo del margen por préstamos (Anual):
# Tasa activa menos Tasa de transferencia interna (fijada en 9.5%).
# Se divide entre 100 porque las tasas están expresadas en porcentaje.
clientes_limpios["margen_prestamo"] = (
    clientes_limpios["saldo_prestamo_prom"] * 
    ((clientes_limpios["tasa_activa_anual"] - TASA_TRANSFERENCIA) / 100)
)

# 3. Cálculo del margen por depósitos (Anual):
# Tasa de transferencia interna (9.5%) menos Tasa pasiva pagada al cliente.
# Se divide entre 100 para convertir el porcentaje a valor decimal.
clientes_limpios["margen_deposito"] = (
    clientes_limpios["saldo_deposito_prom"] * 
    ((TASA_TRANSFERENCIA - clientes_limpios["tasa_pasiva_anual"]) / 100)
)

# 4. Cálculo del ingreso anual por comisiones:
# La columna 'comisiones_mes' está mensualizada, por lo que se multiplica por 12.
clientes_limpios["comisiones_anuales"] = clientes_limpios["comisiones_mes"] * 12

# 5. Cálculo del Ingreso Anual Total por Cliente:
# Suma de los tres componentes: margen de préstamo + margen de depósito + comisiones anuales.
clientes_limpios["ingreso_anual"] = (
    clientes_limpios["margen_prestamo"] + 
    clientes_limpios["margen_deposito"] + 
    clientes_limpios["comisiones_anuales"]
)

# 6. Muestra de métricas globales agregadas:
# Imprime el resumen del ingreso anual de la cartera limpia.
print(f"Ingreso anual total de la cartera: $U {clientes_limpios['ingreso_anual'].sum():,.2f}")
print(f"Ingreso anual promedio por cliente: $U {clientes_limpios['ingreso_anual'].mean():,.2f}")
print(f"Ingreso anual mediano por cliente:  $U {clientes_limpios['ingreso_anual'].median():,.2f}")

# 7. Visualización de primeros registros:
# Muestra una vista previa con los componentes clave para verificar los cálculos.
clientes_limpios[[
    "id_cliente", 
    "segmento", 
    "sucursal", 
    "margen_prestamo", 
    "margen_deposito", 
    "comisiones_anuales", 
    "ingreso_anual"
]].head()



Ingreso anual total de la cartera: $U 131,498,238.85
Ingreso anual promedio por cliente: $U 156,359.38
Ingreso anual mediano por cliente:  $U 39,138.04


,id_cliente,segmento,sucursal,margen_prestamo,margen_deposito,comisiones_anuales,ingreso_anual
0,CL0659,Empresa,SUC03,"100,876.68","16,648.22","89,951.28","207,476.18"
1,CL0169,Personal,SUC11,-0.00,"1,408.35","5,640.12","7,048.47"
2,CL0165,Personal,SUC08,"33,129.04","6,469.72","6,585.36","46,184.12"
3,CL0682,Jubilado,SUC01,-0.00,901.74,"1,719.60","2,621.34"
4,CL0822,Pyme,SUC12,"727,006.57","9,124.68","14,293.68","750,424.93"


In [ ]:
# Caso B: construir facturación neta, margen bruto y margen neto, todo en pesos uruguayos.

<h2 style='font-weight:bold; color:#003366;'>Etapa 4: clasificar y localizar la anomalía <span style='font-weight:normal; font-size:70%;'>(10 minutos)</span></h2>

Ya existe la variable que importa. Ahora hay que clasificar las filas y usar los recuentos para localizar
el origen de lo que salga raro.

1. Para dos categorías alcanza `np.where`. Para tres o más es mejor construirlo con un bucle `for` y estructura condicional de `if` y `elif`.

2. Una vez que la clasificación existe, filtrar las filas problemáticas y
pedir `value_counts()` de alguna columna categórica dice de dónde vienen. Y comparar ese recuento contra
el recuento del total evita la conclusión falsa de que el problema está donde simplemente hay más filas.

**Caso A.** Clasificar los clientes en tramos según el ingreso anual construido en la etapa 3, y después
buscar los préstamos mal tarifados: aquellos donde la tasa que el banco cobra es menor que la tasa que le
paga al mismo cliente por sus depósitos. Cada uno de esos préstamos pierde plata por definición.

**Caso B.** Clasificar las facturas según el descuento otorgado, tomando como referencia la política de
la empresa, que autoriza hasta 15%. Después buscar las facturas con margen neto negativo y ver de qué
línea de producto y de qué vendedor vienen.

In [ ]:
# Caso A
tramos = []

for ingreso in clientes_limpios["ingreso_anual"]:
    if ingreso < 0:
        tramos.append("Destruye valor")
    elif ingreso < 50_000:
        tramos.append("Marginal")
    elif ingreso < 500_000:
        tramos.append("Rentable")
    else:
        tramos.append("Estratégico")

clientes_limpios["tramo_valor"] = tramos

print(clientes_limpios["tramo_valor"].value_counts())

# Préstamos mal tarifados: completar la condición y el recuento
mal_tarifados = clientes_limpios[
    (clientes_limpios["tasa_activa_anual"] < clientes_limpios["tasa_pasiva_anual"]) 
    & (clientes_limpios["saldo_prestamo_prom"] > 0)
]
print("\nPréstamos con spread negativo:", len(mal_tarifados))
print(mal_tarifados["sucursal"].value_counts())
print(f"\nPérdida anual que generan: $U {mal_tarifados['margen_prestamo'].sum():,.0f}")

tramo_valor
Marginal          456
Rentable          318
Estratégico        62
Destruye valor      5
Name: count, dtype: int64

Préstamos con spread negativo: 9
sucursal
SUC14    3
SUC08    2
SUC06    1
SUC01    1
SUC02    1
SUC13    1
Name: count, dtype: int64

Pérdida anual que generan: $U -189,581


In [ ]:
# Caso B
DESCUENTO_AUTORIZADO = 0.15

facturas_limpias["excede_politica"] = np.where(
    facturas_limpias["descuento"] > DESCUENTO_AUTORIZADO, "Si", "No")

print("Facturas por encima de la política de descuentos:")
print(facturas_limpias["excede_politica"].value_counts())

# De quién vienen esas facturas, y comparado contra el peso de cada vendedor en el total
print("\nVendedores en las facturas que exceden la política:")
print(____)

print("\nPeso de cada vendedor en el total de facturas:")
print(____)

# Y las facturas que directamente pierden plata: de qué línea de producto vienen
perdedoras = facturas_limpias[____]
print("\nFacturas con margen neto negativo:", len(perdedoras))
print(perdedoras["linea_producto"].value_counts())

<h2 style='font-weight:bold; color:#003366;'>Etapa 5: conclusión de análisis <span style='font-weight:normal; font-size:70%;'>(8 minutos)</span></h2>

**La regla de las tres frases.**:

1. Qué se encontró, en una oración, con el número adentro.
2. Con qué evidencia, o sea qué cálculo lo respalda y sobre cuántas filas.
3. Qué decisión sugiere, y qué se necesitaría saber para estar seguro.

**Un requisito adicional.** Cada mesa tiene que llevar a la puesta en común un número que contradiga la
hipótesis inicial de quien pidió el análisis. En el caso A la hipótesis del comité es que el problema
son las sucursales del interior. En el caso B la del dueño es que hay que vender más. Si el análisis las
confirma, también sirve, pero hay que poder decirlo con evidencia.

In [ ]:
# El cálculo que respalda el hallazgo principal.
# Conviene que quepa en pantalla y que el número esté impreso con su unidad.



**Hallazgo de la mesa**

1. Qué se encontró:

2. Con qué evidencia:

3. Qué decisión sugiere, y qué falta para estar seguro:

**El número que contradice la hipótesis inicial:**

<h2 style='font-weight:bold; color:#003366;'>Puesta en común</h2>

Quince minutos, dos rondas.

**Primera ronda, siete minutos por mesa.** Dos o tres personas de cada mesa cuentan su hallazgo con la
regla de las tres frases. La otra mesa no vio esos datos, así que el hallazgo tiene que entenderse sin
haber abierto el archivo. Vale mostrar una tabla en pantalla, no vale mostrar código.

**Segunda ronda, la mesa que escucha pregunta.** Tres preguntas que siempre corresponden hacer y que
conviene hacer en voz alta:

1. Ese número, sobre cuántas filas está calculado.
2. Qué decisión de limpieza podría estar sosteniendo el resultado, y qué pasaría si se hubiera decidido
   distinto.
3. Hay otra explicación posible para lo mismo.

**Para cerrar, una pregunta para las dos mesas juntas.** Los dos casos terminan encontrando que el
problema no estaba donde lo buscaba quien pidió el análisis. Cuál es la pieza de información que en los
dos casos faltaba en la tabla principal, y en qué se parecen los dos hallazgos.

In [ ]:
# Caso A: concentración del ingreso
top_a = clientes_limpios.nlargest(10, "ingreso_anual")
print(f"Los 10 mejores clientes aportan el "
      f"{top_a['ingreso_anual'].sum() / clientes_limpios['ingreso_anual'].sum():.1%} "
      f"del ingreso anual de la cartera.")

# Y el contraste que importa: los 10 de mayor saldo de depósitos no son los mismos
top_deposito = set(clientes_limpios.nlargest(10, "saldo_deposito_prom")["id_cliente"])
print("Clientes que están en las dos listas:", len(top_deposito & set(top_a["id_cliente"])))

<h2 style='font-weight:bold; color:#003366; text-align:center;'>Para seguir en casa</h2>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 1 *</h3>

Sobre el caso propio, construir una columna que exprese el resultado principal **en términos relativos**
y no absolutos: en el caso A, el ingreso anual de cada cliente dividido por su saldo total, o sea
depósitos más préstamos. En el caso B, el margen neto de cada factura dividido por su facturación.

La pregunta a contestar en un comentario: el cliente o la factura con el mejor resultado absoluto, es
también el mejor en términos relativos.

In [ ]:
# Tu código acá

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
clientes_limpios["saldo_total"] = (clientes_limpios["saldo_deposito_prom"]
                                   + clientes_limpios["saldo_prestamo_prom"])
clientes_limpios["retorno_sobre_saldo"] = (clientes_limpios["ingreso_anual"]
                                           / clientes_limpios["saldo_total"])

print(clientes_limpios.nlargest(5, "ingreso_anual")[
    ["id_cliente", "segmento", "ingreso_anual", "retorno_sobre_saldo"]])
print(clientes_limpios.nlargest(5, "retorno_sobre_saldo")[
    ["id_cliente", "segmento", "ingreso_anual", "retorno_sobre_saldo"]])

# Caso B
facturas_limpias["margen_pct"] = (facturas_limpias["margen_neto"]
                                  / facturas_limpias["facturacion"])
print(facturas_limpias.nlargest(5, "margen_neto")[
    ["id_factura", "linea_producto", "margen_neto", "margen_pct"]])
print(facturas_limpias.nlargest(5, "margen_pct")[
    ["id_factura", "linea_producto", "margen_neto", "margen_pct"]])
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 2 **</h3>

Los dos casos tienen una variable continua que conviene convertir en tramos para poder hablar de ella:
en el caso A la antigüedad del cliente, que sale de `fecha_alta`. En el caso B los días de cobro, que ya
se calcularon en la etapa 2.

Armar los tramos con cortes que tengan sentido de negocio y no cortes automáticos: en el caso A, menos de
un año, uno a cinco, cinco a diez, más de diez. En el caso B, dentro del plazo, hasta 30 días de atraso,
más de 30, sin cobrar.

Después contestar: los tramos quedaron con cantidades comparables de casos, y si no, eso invalida la
comparación entre tramos.

In [ ]:
# Tu código acá

<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b> (abrila solo después de intentarlo)</summary>

Para cortar una variable continua en tramos definidos por el negocio existe `pd.cut`, que recibe la
columna, la lista de cortes y la lista de etiquetas. La alternativa, `pd.qcut`, corta por cantidad de
casos y no por valor: sirve para otra cosa, para comparar quintiles.

En el caso A, la antigüedad en años sale de restar `fecha_alta` a una fecha de corte y dividir los días
por 365,25. Conviene usar como fecha de corte la última fecha del dataset y no la fecha de hoy, para que
el resultado no cambie según el día en que se corre el notebook.

En el caso B hay un detalle: las facturas sin cobrar tienen `dias_cobro` faltante, y `pd.cut` las va a
dejar afuera. Hay que decidir explícitamente qué hacer con ellas.

</details>

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
FECHA_CORTE = pd.Timestamp("2026-06-30")
clientes_limpios["antiguedad_anios"] = (
    (FECHA_CORTE - clientes_limpios["fecha_alta"]).dt.days / 365.25)

clientes_limpios["tramo_antiguedad"] = pd.cut(
    clientes_limpios["antiguedad_anios"],
    bins=[0, 1, 5, 10, 100],
    labels=["Menos de 1 año", "1 a 5", "5 a 10", "Más de 10"])
print(clientes_limpios["tramo_antiguedad"].value_counts().sort_index())

# Caso B
facturas_limpias["atraso"] = (facturas_limpias["dias_cobro"]
                              - facturas_limpias["plazo_pago_dias"])
facturas_limpias["tramo_cobro"] = pd.cut(
    facturas_limpias["atraso"],
    bins=[-10_000, 0, 30, 10_000],
    labels=["En plazo", "Hasta 30 días", "Más de 30 días"])

# Las no cobradas se etiquetan aparte, porque dejarlas como faltantes las esconde
facturas_limpias["tramo_cobro"] = (facturas_limpias["tramo_cobro"]
                                   .cat.add_categories(["Sin cobrar"])
                                   .fillna("Sin cobrar"))
print(facturas_limpias["tramo_cobro"].value_counts())
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 3 ***</h3>

Volver sobre la decisión de limpieza de la etapa 2 y medir cuánto depende el hallazgo de esa decisión.

Concretamente: rehacer el cálculo del resultado principal de dos formas, una con el criterio elegido y
otra con el criterio opuesto, y reportar las dos cifras juntas. En el caso A, con y sin los duplicados.
En el caso B, descartando y no descartando las facturas sin unidades.

La entrega es una tabla de dos filas con las dos versiones del número, y una frase sobre si la conclusión
del hallazgo se sostiene en las dos.

In [ ]:
# Tu código acá

<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b> (abrila solo después de intentarlo)</summary>

No hace falta ninguna función nueva. Alcanza con calcular el número dos veces sobre dos DataFrames
distintos y armar con los resultados un DataFrame chico, que se puede construir a partir de un
diccionario de listas.

El sentido del ejercicio es el reflejo profesional: cuando un número sostiene una decisión, corresponde
saber cuánto se mueve si la decisión metodológica hubiera sido la contraria. Si se mueve poco, el
hallazgo es robusto. Si se mueve mucho, la decisión de limpieza pasa a ser parte del informe y no una
nota al pie.

</details>

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
def ingreso_anual_total(tabla):
    margen_prestamo = tabla["saldo_prestamo_prom"] * (
        tabla["tasa_activa_anual"].fillna(0) - TASA_TRANSFERENCIA) / 100
    margen_deposito = tabla["saldo_deposito_prom"] * (
        TASA_TRANSFERENCIA - tabla["tasa_pasiva_anual"]) / 100
    return (margen_prestamo + margen_deposito + tabla["comisiones_mes"] * 12).sum()


comparacion = pd.DataFrame({
    "criterio": ["Con duplicados", "Sin duplicados"],
    "ingreso_anual_cartera": [ingreso_anual_total(clientes),
                              ingreso_anual_total(clientes.drop_duplicates())],
})
comparacion["diferencia_pct"] = (comparacion["ingreso_anual_cartera"]
                                 / comparacion["ingreso_anual_cartera"].iloc[1] - 1)
print(comparacion)

# Caso B
cargadas = facturas_limpias["unidades"].notna()
print(pd.DataFrame({
    "criterio": ["Incluyendo facturas sin unidades", "Descartando facturas sin unidades"],
    "facturacion": [facturas_limpias["facturacion"].sum(),
                    facturas_limpias.loc[cargadas, "facturacion"].sum()],
    "facturas": [len(facturas_limpias), int(cargadas.sum())],
    "ticket_promedio": [facturas_limpias["facturacion"].sum() / len(facturas_limpias),
                        facturas_limpias.loc[cargadas, "facturacion"].mean()],
}))

# El resultado tiene una trampa que vale más que el ejercicio: la facturación total da
# exactamente igual en las dos versiones. Es porque un faltante en unidades propaga el
# faltante a la facturación de esa fila, y sum() descarta los faltantes sin avisar.
# O sea que pandas ya tomó la decisión de descartarlas, en silencio. Donde sí aparece la
# diferencia es en el ticket promedio, porque el denominador cambia según cómo se cuenten
# las filas. La conclusión práctica: los promedios son el lugar donde los faltantes muerden.
```

</details>

<h2 style='font-weight:bold; color:#003366;'>Anexo: Panda analítico para recordar</h2>

Referencia rápida de lo que se usó hoy, para tener a mano durante la investigación.

| Qué se quiere | Cómo se escribe |
|---|---|
| Cargar declarando fechas | `pd.read_csv(ruta, parse_dates=["fecha"])` |
| Forma de la tabla | `tabla.shape` |
| Tipos de cada columna | `tabla.dtypes` |
| Faltantes por columna | `tabla.isna().sum()` |
| Resumen numérico | `tabla[["a", "b"]].describe()` |
| Filas duplicadas | `tabla.duplicated().sum()` |
| Identificadores repetidos | `tabla.duplicated(subset="id").sum()` |
| Sacar duplicados | `tabla.drop_duplicates()` |
| Filtrar por una condición | `tabla[tabla["monto"] < 0]` |
| Filtrar por dos condiciones | `tabla[(tabla["a"] > 0) & (tabla["b"] == "Si")]` |
| Elegir filas y columnas | `tabla.loc[tabla["a"] > 0, ["id", "a"]]` |
| Columna nueva | `tabla["c"] = tabla["a"] * tabla["b"]` |
| Reemplazar faltantes | `tabla["a"].fillna(0)` |
| Dos valores según condición | `np.where(condicion, "Si", "No")` |
| Varios valores según condiciones | `np.select([cond1, cond2], ["A", "B"], default="C")` |
| Recuento de categorías | `tabla["categoria"].value_counts()` |
| Recuento en proporción | `tabla["categoria"].value_counts(normalize=True)` |
| Las n filas más altas | `tabla.nlargest(10, "monto")` |
| Ordenar | `tabla.sort_values("monto", ascending=False)` |
| Días entre dos fechas | `(tabla["f2"] - tabla["f1"]).dt.days` |
| Mes de una fecha | `tabla["fecha"].dt.month` |
| Percentil | `tabla["monto"].quantile(0.9)` |

Sobre `SettingWithCopyWarning`: si aparece esa advertencia al crear una columna, es porque se está
escribiendo sobre una porción de otra tabla y pandas no garantiza dónde queda el resultado. Se resuelve
agregando `.copy()` cuando se creó la porción, que es la razón del `.copy()` en `drop_duplicates()` de
la etapa 2.